# Group 32 — ROME baseline
Fresh-session entry point. Choose a GPU runtime, then run cells in order. Setup downloads Python 3.11 and CUDA PyTorch; the notebook kernel can stay on Python 3.13. No notebook runtime downgrade is needed.

The archived run completed 200 facts: efficacy 100%, paraphrase 94.75%, neighborhood 74.45%. This notebook starts a new run. New setup packaging has not yet been executed on a fresh GPU; see README validation limits.


In [ ]:
import os, sys, subprocess, json, shutil
from pathlib import Path
os.environ['MPLBACKEND'] = 'Agg'
ROOT = Path('/content/nlp544')
if not ROOT.exists():
    subprocess.run(['git','clone','https://github.com/rohithreddy009/nlp544.git',str(ROOT)],check=True)

def stream(command):
    with subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as p:
        for line in p.stdout:
            print(line,end='',flush=True)
        if p.wait():
            raise RuntimeError('Command failed; see output above.')

stream([sys.executable,'-u',str(ROOT/'scripts/setup.py')])
PYTHON = str(ROOT/'.venv/bin/python')


## Persistent results
Authorize Drive when prompted. Keep a new run separate from the archived results.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = Path('/content/drive/MyDrive/Group32/ROME_baseline/gpt2xl_cf200_repo_v1')
OUT.mkdir(parents=True,exist_ok=True)

def run_stage(limit):
    stream([PYTHON,'-u',str(ROOT/'scripts/run_stage.py'),'--limit',str(limit),'--output',str(OUT)])
    summary=json.loads((OUT/f'summary_{limit}.json').read_text())
    print(json.dumps({k:v for k,v in summary.items() if k!='case_ids'},indent=2))
    return summary


## Run 1 facts
Completed cases are reused. Progress prints after every newly completed case.

In [ ]:
summary = run_stage(1)


## Run 5 facts
Completed cases are reused. Progress prints after every newly completed case.

In [ ]:
summary = run_stage(5)


## Run 200 facts
Completed cases are reused. Progress prints after every newly completed case.

In [ ]:
summary = run_stage(200)


## Download results

In [ ]:
assert json.loads((OUT/'summary_200.json').read_text())['complete_200']
archive=shutil.make_archive('/content/Group32_ROME_Results','zip',root_dir=OUT)
from google.colab import files
files.download(archive)
